In [36]:
# !pip install --upgrade --force-reinstall --no-cache-dir numpy scipy scikit-learn gensim

In [37]:

# !python -m spacy download en_core_web_md

# 1. Import dan Load Dataset

In [38]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
import spacy
import nltk
from nltk.corpus import stopwords
import string
nlp = spacy.load("en_core_web_md")


In [39]:
import io, zipfile, requests
import pandas as pd

url = "https://drive.google.com/uc?export=download&id=1edTm4a2u_--1OMV0wmKbDMpt82EJCQMN"

r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))

train_name = next(n for n in z.namelist() if n.endswith("train.csv"))
test_name  = next(n for n in z.namelist() if n.endswith("test.csv"))

train_df = pd.read_csv(z.open(train_name))
test_df  = pd.read_csv(z.open(test_name))

print(train_df.shape, test_df.shape)


(586, 3) (146, 2)


# 2. Pre-Processing
Tahap ini bertujuan untuk membersihkan dan menyiapkan teks mentah agar bisa diolah oleh model machine learning.
Langkah-langkah yang dilakukan meliputi:
- **Lowercasing** untuk menyeragamkan bentuk kata.
- **Normalisasi slang/akronim** agar kata informal diganti dengan bentuk baku.
- **Tokenisasi** untuk memecah kalimat menjadi kata-kata (token).
- **Koreksi ejaan** guna mengurangi variasi kata yang tidak perlu.
- **Penanganan negasi** (misalnya "not good" → "not_good") agar makna sentimen tidak hilang.
- **Stopword removal** dengan pengecualian kata penting seperti *not*, *but*, atau *very*.
- **Lemmatization** untuk mengembalikan kata ke bentuk dasar.  

Proses ini penting untuk mengurangi *noise* sekaligus menjaga informasi yang relevan untuk analisis sentimen.

In [40]:
# !pip install pyspellchecker
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from spellchecker import SpellChecker
import nltk
import re

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("punkt_tab")
nltk.download("omw-1.4")
nltk.download("stopwords")

# Spelling Correction
spell = SpellChecker()
# Normalize Slang
slang_dict = {
    "btw": "by the way", "omg": "oh my god", "idk": "i don't know",
    "u": "you", "r": "are", "thx": "thanks", "plz": "please"
}
def normalize_slang(text, slang_dict):
    for word, replacement in slang_dict.items():
        text = re.sub(r'\b' + re.escape(word) + r'\b', replacement, text)
    return text

# Custom Stopword Removal
stop_words = set(stopwords.words("english"))
words_to_keep = {
    'not', 'no', 'never', 'n\'t',
    'but', 'however',
    'very', 'so', 'too', 'extremely', 'quite'
}
stop_words = stop_words - words_to_keep

# Lemmatization
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, lang='en', remove_stopwords=True, keep_negation=True):
    if pd.isnull(text):
        return ""
    # Lowercase
    text = text.lower()
    # Normalize Slang
    text = normalize_slang(text, slang_dict)
    # Tokenization
    tokens = word_tokenize(text)
    # Spelling Correction
    misspelled = spell.unknown(tokens)
    tokens = [spell.correction(word) if word in misspelled and spell.correction(word) is not None else word for word in tokens]
    # Negation Handling
    if keep_negation:
        negations = {"not","no","never","n't","cannot","cant",
                     "don't","doesn't","didn't","isn't","wasn't",
                     "won't","shouldn't","without"}
        out = []
        i = 0
        while i < len(tokens):
            t = tokens[i]
            if t in negations and i+1 < len(tokens):
                out.append("not_" + tokens[i+1])
                i += 2
            else:
                out.append(t)
                i += 1
        tokens = out
    # Stopword Removal
    if remove_stopwords:
        tokens = [w for w in tokens if (w not in stop_words) or w.startswith("not_")]
    # Lemmatization
    if lang == 'en':
        tokens = [lemmatizer.lemmatize(w) for w in tokens if w.isalpha()]


    return " ".join(tokens)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [41]:
# !pip install tqdm -q
# !pip install swifter

from tqdm import tqdm
tqdm.pandas(desc="Processing Text")
import swifter # For parallism

train_df["clean_stop"] = train_df["Text"].astype(str).swifter.apply(lambda x: preprocess_text(x, remove_stopwords=True))
train_df["clean_nostop"] = train_df["Text"].astype(str).swifter.apply(lambda x: preprocess_text(x, remove_stopwords=False))

Pandas Apply:   0%|          | 0/586 [00:00<?, ?it/s]

Pandas Apply:   0%|          | 0/586 [00:00<?, ?it/s]

In [42]:
# Encode Label
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_df["Sentiment_encoded"] = le.fit_transform(train_df["Sentiment"])

# 3. Feature Extraction
Setelah teks dibersihkan melalui tahap preprocessing, langkah berikutnya adalah mengubah teks menjadi representasi numerik
agar dapat diproses oleh algoritma machine learning. Proses ini disebut **feature extraction**.  

Beberapa metode yang digunakan dalam eksperimen ini antara lain:
- **Bag of Words (BoW)**: menghitung frekuensi kemunculan kata tanpa memperhatikan urutan.
- **TF-IDF (Term Frequency – Inverse Document Frequency)**: memberi bobot pada kata berdasarkan seberapa sering muncul di suatu dokumen
  dan seberapa jarang muncul di seluruh korpus.
- **Word Embedding (Word2Vec, SBERT)**: memetakan kata atau kalimat ke vektor padat berdimensi rendah yang
  mampu menangkap makna semantik.
- **Dimensionality Reduction (SVD)**: mereduksi jumlah dimensi fitur agar lebih efisien untuk diproses.  

Tahap ini penting karena model machine learning tidak bisa langsung memahami teks mentah, sehingga teks perlu diubah ke bentuk numerik
yang tetap merepresentasikan informasi semantik maupun pola penting dalam dokumen.

In [43]:
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sentence_transformers import SentenceTransformer

sbert = SentenceTransformer('all-MiniLM-L6-v2')
def sbert_features(texts):
    return sbert.encode(texts, convert_to_numpy=True)

In [44]:
def extract_features(name, texts):
    if name == "SBERT":
        return sbert.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    else:
        return feature_extractors[name].fit_transform(texts)

# 4. Shallow ML
Pada tahap ini, data yang sudah direpresentasikan dalam bentuk numerik
akan diproses menggunakan algoritma **shallow machine learning** untuk melakukan klasifikasi sentimen.  
Shallow ML mengacu pada model pembelajaran mesin yang relatif sederhana, tidak terlalu dalam (berbeda dengan deep learning),
namun tetap efektif untuk data teks.  

Beberapa algoritma yang diuji dalam eksperimen ini antara lain:
- **Naive Bayes**: model probabilistik sederhana berbasis Teorema Bayes.
- **K-Nearest Neighbor (KNN)**: mengklasifikasikan teks berdasarkan kedekatannya dengan dokumen lain.
- **Support Vector Machine (SVM)**: mencari *hyperplane* dengan margin maksimal untuk memisahkan kelas.
- **Random Forest & XGBoost**: model *ensemble* berbasis pohon keputusan yang kuat dalam menangkap pola non-linear.

Tahap ini bertujuan untuk membandingkan performa berbagai algoritma *shallow learning*
dan menentukan mana yang paling efektif untuk tugas klasifikasi sentimen.

In [45]:
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Gaussian Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "KNN": KNeighborsClassifier(n_neighbors=20),
    "SVM": SVC(gamma="scale",kernel="linear", probability=False),
    "Random Forest": RandomForestClassifier(n_estimators=500, random_state=42),
    "LinearSVC": LinearSVC(max_iter=5000),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
}


# 5. Main Process
Tahap ini merupakan inti dari eksperimen klasifikasi sentimen.
Pada bagian ini dilakukan proses pelatihan (training) model dengan data latih,
serta evaluasi menggunakan data validasi untuk mengukur performa.  

Proses utama mencakup:
1. **Training** – model *shallow machine learning* dilatih menggunakan fitur hasil ekstraksi (BoW, TF-IDF, Word2Vec, dll.).  
2. **Validation** – performa model dievaluasi dengan metrik seperti **Macro F1-score** pada data validasi.  
   Validation penting agar kita dapat membandingkan kombinasi preprocessing, ekstraksi fitur, dan algoritma,
   sekaligus mencegah *overfitting*.  

Output dari tahap ini adalah tabel atau laporan skor performa dari tiap kombinasi,
yang menjadi dasar analisis.

In [46]:
from gensim.models import Word2Vec
results = []

preprocess_variants = {
    "with_stopwords": train_df["clean_stop"].fillna("").astype(str),
    "no_stopwords": train_df["clean_nostop"].fillna("").astype(str)
}

for prep_name, prep_texts in preprocess_variants.items():
    X_train, X_val, y_train, y_val = train_test_split(
        prep_texts, train_df["Sentiment_encoded"],
        test_size=0.2, random_state=42, stratify=train_df["Sentiment_encoded"]
    )

    # Word2Vec
    tokenized_train = [text.split() for text in X_train]
    w2v_model = Word2Vec(sentences=tokenized_train, vector_size=100, window=5, min_count=1, workers=4)
    w2v_model.train(tokenized_train, total_examples=len(tokenized_train), epochs=10)

    def w2v_features(texts, model):
        vectors = []
        for text in texts:
            words = text.split()
            word_vectors = [model.wv[word] for word in words if word in model.wv]
            if not word_vectors:
                vectors.append(np.zeros(model.vector_size))
            else:
                vectors.append(np.mean(word_vectors, axis=0))
        return np.array(vectors)

    # Feature Extraction
    feature_extractors = {
        "BoW": CountVectorizer(),
        "TFIDF_unigram": TfidfVectorizer(ngram_range=(1,1)),
        "TFIDF_bigram": TfidfVectorizer(ngram_range=(1,2)),
        "BoW_SVD": Pipeline([             #
            ("bow", CountVectorizer()),
            ("svd", TruncatedSVD(n_components=200, random_state=42))
        ]),
        "TFIDF_bigram_SVD": Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1,2), max_features=20000)),
            ("svd", TruncatedSVD(n_components=200, random_state=42))
        ]),
        "SBERT":  sbert_features,
        "Word2Vec": lambda texts: w2v_features(texts, w2v_model)
    }

    for feat_name, extractor in feature_extractors.items():
        try:
            if feat_name in ["SBERT", "Word2Vec"]:
                X_train_vec = extractor(X_train.tolist())
                X_val_vec   = extractor(X_val.tolist())
            else:
                extractor.fit(X_train)
                X_train_vec = extractor.transform(X_train)
                X_val_vec   = extractor.transform(X_val)

            for model_name, model in models.items():
                if feat_name in ["SBERT", "Word2Vec", "TFIDF_bigram_SVD","BoW_SVD"] and model_name == "Multinomial Naive Bayes":
                    continue

                if feat_name in ["BoW", "TFIDF_unigram"] and model_name == "Gaussian Naive Bayes":
                    continue
                try:
                    X_train_in, X_val_in = X_train_vec, X_val_vec

                    if model_name == "Gaussian Naive Bayes" and hasattr(X_train_in, "toarray"):
                        X_train_in = X_train_in.toarray()
                        X_val_in   = X_val_in.toarray()

                    model.fit(X_train_in, y_train)
                    preds = model.predict(X_val_in)
                    report = classification_report(y_val, preds, output_dict=True, zero_division=0)

                    # Process Pipeline
                    results.append({
                        "Preprocessing": prep_name,
                        "Feature": feat_name,
                        "Model": model_name,
                        "Macro F1": report["macro avg"]["f1-score"]
                    })
                    print(f"Process Done: {prep_name} + {feat_name} + {model_name}")

                except Exception as e:
                    print(f"Error {model_name}: {e}")

        except Exception as e:
            print(f"Error {feat_name}: {e}")

# Comparation in table
results_df = pd.DataFrame(results).sort_values(by="Macro F1", ascending=False)
display(results_df)

Process Done: with_stopwords + BoW + Multinomial Naive Bayes
Process Done: with_stopwords + BoW + Logistic Regression
Process Done: with_stopwords + BoW + KNN
Process Done: with_stopwords + BoW + SVM
Process Done: with_stopwords + BoW + Random Forest
Process Done: with_stopwords + BoW + LinearSVC
Process Done: with_stopwords + BoW + XGBoost
Process Done: with_stopwords + TFIDF_unigram + Multinomial Naive Bayes
Process Done: with_stopwords + TFIDF_unigram + Logistic Regression
Process Done: with_stopwords + TFIDF_unigram + KNN


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [09:57:27] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Process Done: with_stopwords + TFIDF_unigram + SVM
Process Done: with_stopwords + TFIDF_unigram + Random Forest
Process Done: with_stopwords + TFIDF_unigram + LinearSVC
Process Done: with_stopwords + TFIDF_unigram + XGBoost
Process Done: with_stopwords + TFIDF_bigram + Multinomial Naive Bayes


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [09:57:29] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Process Done: with_stopwords + TFIDF_bigram + Gaussian Naive Bayes
Process Done: with_stopwords + TFIDF_bigram + Logistic Regression
Process Done: with_stopwords + TFIDF_bigram + KNN
Process Done: with_stopwords + TFIDF_bigram + SVM
Process Done: with_stopwords + TFIDF_bigram + Random Forest
Process Done: with_stopwords + TFIDF_bigram + LinearSVC


...

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [09:58:31] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Process Done: no_stopwords + SBERT + XGBoost
Process Done: no_stopwords + Word2Vec + Gaussian Naive Bayes
Process Done: no_stopwords + Word2Vec + Logistic Regression
Process Done: no_stopwords + Word2Vec + KNN
Process Done: no_stopwords + Word2Vec + SVM
Process Done: no_stopwords + Word2Vec + Random Forest
Process Done: no_stopwords + Word2Vec + LinearSVC


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [09:58:36] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Process Done: no_stopwords + Word2Vec + XGBoost


,Preprocessing,Feature,Model,Macro F1
10,with_stopwords,TFIDF_unigram,SVM,0.832887
4,with_stopwords,BoW,Random Forest,0.832887
6,with_stopwords,BoW,XGBoost,0.819240
11,with_stopwords,TFIDF_unigram,Random Forest,0.797390
12,with_stopwords,TFIDF_unigram,LinearSVC,0.795751
...,...,...,...,...
16,with_stopwords,TFIDF_bigram,Logistic Regression,0.456221
57,no_stopwords,TFIDF_unigram,Multinomial Naive Bayes,0.456221
29,with_stopwords,TFIDF_bigram_SVD,Gaussian Naive Bayes,0.313510
72,no_stopwords,BoW_SVD,Gaussian Naive Bayes,0.233766


# 6. Submission untuk Kaggle

In [47]:
best_result = results_df.iloc[0]
best_prep_name = best_result['Preprocessing']
best_feat_name = best_result['Feature']
best_model_name = best_result['Model']

X_full = preprocess_variants[best_prep_name]
is_stopword_removed = (best_prep_name == 'with_stopwords')
test_df["clean_text"] = test_df["Text"].astype(str).apply(lambda x: preprocess_text(x, remove_stopwords=is_stopword_removed))
X_test = test_df['clean_text']

if "SVD" in best_feat_name:
    n_components = 200
    if "BoW" in best_feat_name:
        base_extractor = Pipeline([("bow", CountVectorizer()), ("svd", TruncatedSVD(n_components=n_components))])
    else:
        base_extractor = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2))), ("svd", TruncatedSVD(n_components=n_components))])
elif best_feat_name == "SBERT" or best_feat_name == "Word2Vec":
    tokenized_full = [text.split() for text in X_full]
    w2v_model_final = Word2Vec(sentences=tokenized_full, vector_size=100, window=5, min_count=1, workers=4)
    w2v_model_final.train(tokenized_full, total_examples=len(tokenized_full), epochs=10)
    final_feature_extractors = {
        "SBERT": sbert_features,
        "Word2Vec": lambda texts: w2v_features(texts, w2v_model_final)
    }
    base_extractor = final_feature_extractors[best_feat_name]
else:
    base_extractor = feature_extractors[best_feat_name]

if callable(base_extractor):
    X_full_vec = base_extractor(X_full.tolist())
    X_test_vec = base_extractor(X_test.tolist())
else:
    base_extractor.fit(X_full)
    X_full_vec = base_extractor.transform(X_full)
    X_test_vec = base_extractor.transform(X_test)

best_model = models[best_model_name]

X_train_final = X_full_vec
if "Gaussian" in best_model_name and hasattr(X_train_final, "toarray"):
    X_train_final = X_train_final.toarray()

best_model.fit(X_train_final, train_df["Sentiment_encoded"])

preds = best_model.predict(X_test_vec)
preds_labels = le.inverse_transform(preds)

submission_df = pd.DataFrame({"id": test_df["id"], "Sentiment": preds_labels})
submission_df.to_csv('submission_final.csv', index=False)

display(submission_df.head())

,id,Sentiment
0,1,Negative
1,2,Positive
2,3,Negative
3,4,Negative
4,5,Positive
